In [0]:
import sys
sys.path.append("../")

In [0]:
os.getcwd()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import DoubleType

from src.metadata_reader import read_yaml
from src.spark_utils import get_spark, save_delta

# -----------------------------------
# Create Spark Session
# -----------------------------------

spark = get_spark()

# -----------------------------------
# Read Metadata
# -----------------------------------

config = read_yaml(
    "../metadata/silver_config.yaml"
)

# -----------------------------------
# Read Bronze Table
# -----------------------------------

df = spark.table(
    config["source"]["table"]
)

# -----------------------------------
# Metadata Driven Cleaning
# -----------------------------------

for rule in config["cleaning_rules"]:

    column = rule["column"]
    action = rule["action"]

    if action == "trim":

        df = df.withColumn(
            column,
            trim(col(column))
        )

    elif action == "upper":

        df = df.withColumn(
            column,
            upper(col(column))
        )

    elif action == "lower":

        df = df.withColumn(
            column,
            lower(col(column))
        )

    elif action == "cast_double":  # Add trimming before casting            
        df = df.withColumn(
            column,
            trim(col(column)).cast(DoubleType())
        )


        df = df.withColumn(
            column,
            col(column).cast(DoubleType())
        )

# -----------------------------------
# Custom Business Logic
# -----------------------------------

# df = (
#     df.withColumn(
#         "price",
#         col("price").cast(DoubleType())
#     )
# )

# df = df.withColumn(
#     "price_category",
#     when(col("price") >= 1000, "Premium")
#     .when(col("price") >= 500, "Standard")
#     .otherwise("Budget")
# )

# -----------------------------------
# Save Silver Table
# -----------------------------------

save_delta(
    df,
    config["target"]["database"],
    config["target"]["table"]
)